# DeepSeek-OCR-2

In [1]:
!nvidia-smi


Sun Mar  8 20:57:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   75C    P0             24W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.46.3 tokenizers==0.20.3
!pip install einops addict easydict
!pip install pymupdf pillow
!pip install flash-attn==2.7.3 --no-build-isolation

Looking in indexes: https://download.pytorch.org/whl/cu118


In [3]:
import torch
import transformers
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


PyTorch: 2.6.0+cu118
Transformers: 4.46.3
CUDA available: True
GPU: NVIDIA L4


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from transformers import AutoModel, AutoTokenizer
import torch

MODEL_NAME = "deepseek-ai/DeepSeek-OCR-2"

print("Laddar tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Laddar modell (kan ta några minuter)...")
model = AutoModel.from_pretrained(
    MODEL_NAME,
    _attn_implementation="flash_attention_2",
    trust_remote_code=True,
    use_safetensors=True
)
model = model.eval().cuda().to(torch.bfloat16)
print(" Modell laddad!")

Laddar tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Laddar modell (kan ta några minuter)...


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


 Modell laddad!


In [ ]:
try:
    from google.colab import files
    print("Upload your PDF file:")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"PDF uploaded: {pdf_path}")
    OUTPUT_DIR = "/content/ocr_output"
except ImportError:
    pdf_path = "path/to/local/file.pdf"
    print(f"Running locally, using: {pdf_path}")
    OUTPUT_DIR = "./ocr_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

Välj din PDF-fil:


Saving FlexQube.pdf to FlexQube (1).pdf

 Uppladdad: FlexQube (1).pdf


In [12]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path: str, dpi: int = 200) -> list[str]:
    doc = fitz.open(pdf_path)
    image_paths = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        zoom = dpi / 72
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)

        img_path = f"/content/page_{page_num + 1}.png"
        pix.save(img_path)
        image_paths.append(img_path)
        print(f"  Sida {page_num + 1}/{len(doc)} konverterad")

    doc.close()
    return image_paths

print(f"Konverterar {pdf_filename} till bilder...")
page_images = pdf_to_images(pdf_path)
print(f"\n {len(page_images)} sidor konverterade")

Konverterar FlexQube (1).pdf till bilder...
  Sida 1/68 konverterad
  Sida 2/68 konverterad
  Sida 3/68 konverterad
MuPDF error: format error: No common ancestor in structure tree

  Sida 4/68 konverterad
  Sida 5/68 konverterad
  Sida 6/68 konverterad
  Sida 7/68 konverterad
  Sida 8/68 konverterad
  Sida 9/68 konverterad
  Sida 10/68 konverterad
  Sida 11/68 konverterad
  Sida 12/68 konverterad
  Sida 13/68 konverterad
  Sida 14/68 konverterad
  Sida 15/68 konverterad
  Sida 16/68 konverterad
  Sida 17/68 konverterad
  Sida 18/68 konverterad
  Sida 19/68 konverterad
  Sida 20/68 konverterad
  Sida 21/68 konverterad
  Sida 22/68 konverterad
  Sida 23/68 konverterad
  Sida 24/68 konverterad
  Sida 25/68 konverterad
  Sida 26/68 konverterad
  Sida 27/68 konverterad
  Sida 28/68 konverterad
  Sida 29/68 konverterad
  Sida 30/68 konverterad
  Sida 31/68 konverterad
  Sida 32/68 konverterad
  Sida 33/68 konverterad
  Sida 34/68 konverterad
  Sida 35/68 konverterad
  Sida 36/68 konverterad


In [13]:
import re
from pathlib import Path
from typing import Literal

def run_ocr(
    image_path: str,
    mode: Literal["markdown", "ocr", "free", "figure", "describe"] = "markdown"
) -> str:
    prompts = {
        "markdown": "<image>\n<|grounding|>Convert the document to markdown.",
        "ocr": "<image>\n<|grounding|>OCR this image.",
        "free": "<image>\nFree OCR.",
        "figure": "<image>\nParse the figure.",
        "describe": "<image>\nDescribe this image in detail."
    }
    result = model.infer(
        tokenizer,
        prompt=prompts[mode],
        image_file=image_path,
        output_path=OUTPUT_DIR,
        base_size=1024,
        image_size=768,
        crop_mode=True,
        save_results=False,
        eval_mode=True
    )
    if result is None:
        raise ValueError(f"model.infer() returnerade None för: {image_path}")
    return result

def clean_output(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', text, flags=re.DOTALL)
    text = re.sub(r'<\|det\|>.*?<\|/det\|>', '', text, flags=re.DOTALL)
    text = re.sub(r'<\|.*?\|>', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

OCR_MODE = "markdown"
out_path = Path("/content/output.md")
print(f"Kör OCR i '{OCR_MODE}'-läge...\n")

with out_path.open("w", encoding="utf-8") as f:
    for i, img_path in enumerate(page_images):
        print(f"Processar sida {i + 1}/{len(page_images)}...", end=" ")
        result = run_ocr(img_path, mode=OCR_MODE)
        if i > 0:
            f.write("\n\n---\n\n")
        f.write(f"# Page {i+1}\n\n{clean_output(result)}")
        print("✓")

print(f"\nSparad: {out_path} ({out_path.stat().st_size:,} bytes)")

Kör OCR i 'markdown'-läge...

Processar sida 1/68... 

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Cal

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
✓
Processar sida 2/68... 

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
✓
Processar sida 3/68... 

...

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
✓
Processar sida 67/68... 

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


✓
Processar sida 68/68... =====================
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
✓

Sparad: /content/output.md (221,284 bytes)
